In [ ]:
#COMBINED  SPEI-SR vs DIRECT SOIL+SWE COMPARISON
#A. Corrected SPEI-SR-3
#   Standardized residual based on:
#     R = P - AET - Q
#  This represents a STORAGE-CHANGE / STORAGE-TENDENCY diagnostic.
#B. Direct TerraClimate storage state:
#      S = soil + SWE
# This represents modeled soil-plus-snow STORAGE STATE.


from pathlib import Path
import warnings
import gc

import numpy as np
import pandas as pd
import xarray as xr

from scipy.stats import t as student_t

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import geopandas as gpd

warnings.filterwarnings("ignore")


SPEI_FP = Path(
    r"F:\Historical data\Derived"
    r"\SPEI_SR"
    r"\TerraClimate_SPEI_SR_0p5deg_1979_2023.nc"
)

SPEI_VAR = "SPEI_SR_3"


STATE_TREND_FP = Path(
    r"F:\Historical data\Derived"
    r"\DIRECT_STORAGE_STATE_TRENDS_1979_2023"
    r"\TerraClimate_soilSWE_direct_state_trends_1979_2000_2001_2023.nc"
)


LAND_AREA_FP = Path(
    r"F:\Historical data\Derived"
    r"\DIRECT_STORAGE_STATE_TRENDS_1979_2023"
    r"\CORRECTED_AREA"
    r"\TerraClimate_actual_land_area_0p5deg.nc"
)


WORLD_SHP = Path(
    r"D:\Global\"
    r"\World_Countries\World_Countries_Generalized.shp"
)



# Output directory


OUT_DIR = Path(
    r"F:\Historical data\Derived"
    r"\DIRECT_STORAGE_STATE_TRENDS_1979_2023"
    r"\SPEI_SR_vs_DIRECT_STATE_COMPARISON"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# SPEI-SR trend cache

SPEI_TREND_CACHE = (
    OUT_DIR /
    "Corrected_SPEI_SR3_AR1_trends_1979_2000_2001_2023.nc"
)


P1_START = 1979
P1_END   = 2000

P2_START = 2001
P2_END   = 2023

ALPHA = 0.05

MIN_YEARS_P1 = 22
MIN_YEARS_P2 = 23

RECALCULATE_SPEI_TRENDS = True

DPI = 400


plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 10


print("=" * 80)
print("CHECKING INPUT FILES")
print("=" * 80)

for fp in [
    SPEI_FP,
    STATE_TREND_FP,
    LAND_AREA_FP,
]:

    if not fp.exists():

        raise FileNotFoundError(
            f"Missing file:\n{fp}"
        )

    print(
        "FOUND:",
        fp
    )

# BENJAMINI-HOCHBERG FDR


def bh_fdr_qvalues(
    pvalues
):
    """
    Benjamini-Hochberg FDR q-values.
    """

    p = np.asarray(
        pvalues,
        dtype=np.float64
    )

    output = np.full(
        p.shape,
        np.nan,
        dtype=np.float64
    )

    flat = p.ravel()

    valid = np.isfinite(
        flat
    )

    pv = flat[
        valid
    ]

    m = len(
        pv
    )

    if m == 0:

        return output


    order = np.argsort(
        pv
    )

    p_sorted = pv[
        order
    ]

    rank = np.arange(
        1,
        m + 1,
        dtype=np.float64
    )


    q_sorted = (
        p_sorted
        *
        m
        /
        rank
    )


    q_sorted = np.minimum.accumulate(
        q_sorted[::-1]
    )[::-1]


    q_sorted = np.clip(
        q_sorted,
        0.0,
        1.0
    )


    q_unsorted = np.empty_like(
        q_sorted
    )

    q_unsorted[
        order
    ] = q_sorted


    temp = np.full(
        flat.shape,
        np.nan,
        dtype=np.float64
    )

    temp[
        valid
    ] = q_unsorted


    return temp.reshape(
        p.shape
    )



# PRAIS-WINSTEN AR(1) TREND


def prais_winsten_trend(
    data,
    years,
    min_years
):
    """
    AR(1)-corrected linear trend.

    Parameters
    ----------
    data
        [year, lat, lon]

    years
        array of calendar years

    Returns
    -------
    slope
        units per decade

    pvalue
        AR(1)-corrected p-value

    rho
        AR1 coefficient

    nobs
        annual observations
    """

    Y = np.asarray(
        data,
        dtype=np.float64
    )

    years = np.asarray(
        years,
        dtype=np.float64
    )


    nt, ny, nx = Y.shape

    npix = (
        ny
        *
        nx
    )


    Y2 = Y.reshape(
        nt,
        npix
    )


    finite = np.isfinite(
        Y2
    )

    nobs = finite.sum(
        axis=0
    )


    # Complete annual record required
    usable = (
        (nobs == nt)
        &
        (nobs >= min_years)
    )


    print(
        f"Usable complete pixels = "
        f"{usable.sum():,}"
    )


    slope_output = np.full(
        npix,
        np.nan
    )

    p_output = np.full(
        npix,
        np.nan
    )

    rho_output = np.full(
        npix,
        np.nan
    )


    if not np.any(
        usable
    ):

        raise RuntimeError(
            "No complete annual SPEI-SR records found."
        )


    Yu = Y2[
        :,
        usable
    ]


    n = Yu.shape[
        0
    ]

    x = years


    # Initial OLS
   

    x_mean = np.mean(
        x
    )

    xc = (
        x
        -
        x_mean
    )

    Sxx = np.sum(
        xc ** 2
    )


    y_mean = np.mean(
        Yu,
        axis=0
    )


    slope_ols = (
        np.sum(
            xc[:, None]
            *
            (
                Yu
                -
                y_mean[
                    None,
                    :
                ]
            ),
            axis=0
        )
        /
        Sxx
    )


    intercept_ols = (
        y_mean
        -
        slope_ols
        *
        x_mean
    )


    fitted_ols = (
        intercept_ols[
            None,
            :
        ]
        +
        slope_ols[
            None,
            :
        ]
        *
        x[
            :,
            None
        ]
    )


    residual = (
        Yu
        -
        fitted_ols
    )


    
    # AR(1)
    

    numerator = np.sum(
        residual[
            1:,
            :
        ]
        *
        residual[
            :-1,
            :
        ],
        axis=0
    )


    denominator = np.sum(
        residual[
            :-1,
            :
        ] ** 2,
        axis=0
    )


    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):

        rho = (
            numerator
            /
            denominator
        )


    rho = np.where(
        np.isfinite(
            rho
        ),
        rho,
        0.0
    )


    rho = np.clip(
        rho,
        -0.95,
        0.95
    )


  
    # Prais-Winsten transformation
   

    m = Yu.shape[
        1
    ]


    yt = np.empty_like(
        Yu
    )

    xt = np.empty(
        (
            n,
            m
        ),
        dtype=np.float64
    )

    ct = np.empty(
        (
            n,
            m
        ),
        dtype=np.float64
    )


    first_weight = np.sqrt(
        1.0
        -
        rho ** 2
    )


    yt[
        0,
        :
    ] = (
        first_weight
        *
        Yu[
            0,
            :
        ]
    )


    xt[
        0,
        :
    ] = (
        first_weight
        *
        x[
            0
        ]
    )


    ct[
        0,
        :
    ] = first_weight


    for i in range(
        1,
        n
    ):

        yt[
            i,
            :
        ] = (
            Yu[
                i,
                :
            ]
            -
            rho
            *
            Yu[
                i - 1,
                :
            ]
        )


        xt[
            i,
            :
        ] = (
            x[
                i
            ]
            -
            rho
            *
            x[
                i - 1
            ]
        )


        ct[
            i,
            :
        ] = (
            1.0
            -
            rho
        )


 
    # Transformed regression
    

    A = np.sum(
        ct
        *
        ct,
        axis=0
    )

    B = np.sum(
        ct
        *
        xt,
        axis=0
    )

    C = np.sum(
        xt
        *
        xt,
        axis=0
    )

    D = np.sum(
        ct
        *
        yt,
        axis=0
    )

    E = np.sum(
        xt
        *
        yt,
        axis=0
    )


    determinant = (
        A
        *
        C
        -
        B ** 2
    )


    valid_det = (
        np.isfinite(
            determinant
        )
        &
        (
            np.abs(
                determinant
            )
            >
            1e-12
        )
    )


    intercept = np.full(
        m,
        np.nan
    )

    slope = np.full(
        m,
        np.nan
    )


    intercept[
        valid_det
    ] = (
        D[
            valid_det
        ]
        *
        C[
            valid_det
        ]
        -
        B[
            valid_det
        ]
        *
        E[
            valid_det
        ]
    ) / determinant[
        valid_det
    ]


    slope[
        valid_det
    ] = (
        A[
            valid_det
        ]
        *
        E[
            valid_det
        ]
        -
        B[
            valid_det
        ]
        *
        D[
            valid_det
        ]
    ) / determinant[
        valid_det
    ]


   
    # Standard errors
   

    fitted = (
        ct
        *
        intercept[
            None,
            :
        ]
        +
        xt
        *
        slope[
            None,
            :
        ]
    )


    residual_t = (
        yt
        -
        fitted
    )


    rss = np.sum(
        residual_t ** 2,
        axis=0
    )


    dof = (
        n
        -
        2
    )


    sigma2 = (
        rss
        /
        dof
    )


    var_slope = np.full(
        m,
        np.nan
    )


    var_slope[
        valid_det
    ] = (
        sigma2[
            valid_det
        ]
        *
        A[
            valid_det
        ]
        /
        determinant[
            valid_det
        ]
    )


    se_slope = np.sqrt(
        np.maximum(
            var_slope,
            0.0
        )
    )


    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):

        t_stat = (
            slope
            /
            se_slope
        )


    pvalue = (
        2.0
        *
        student_t.sf(
            np.abs(
                t_stat
            ),
            df=dof
        )
    )


   

    slope_output[
        usable
    ] = (
        slope
        *
        10.0
    )

    p_output[
        usable
    ] = pvalue

    rho_output[
        usable
    ] = rho


    return {
        "slope":
            slope_output.reshape(
                ny,
                nx
            ),

        "pvalue":
            p_output.reshape(
                ny,
                nx
            ),

        "rho":
            rho_output.reshape(
                ny,
                nx
            ),

        "nobs":
            nobs.reshape(
                ny,
                nx
            ),
    }



# 6. CALCULATE---SPEI-SR TRENDS


if (
    RECALCULATE_SPEI_TRENDS
    or
    not SPEI_TREND_CACHE.exists()
):

    print("\n" + "=" * 80)
    print("CALCULATING CORRECTED SPEI-SR-3 TRENDS")
    print("=" * 80)


    spei_ds = xr.open_dataset(
        SPEI_FP,
        decode_times=True
    )


    if SPEI_VAR not in spei_ds.data_vars:

        raise KeyError(
            f"{SPEI_VAR} not found.\n"
            f"Available variables:\n"
            f"{list(spei_ds.data_vars)}"
        )


    spei = (
        spei_ds[
            SPEI_VAR
        ]
        .sel(
            time=slice(
                "1979-01-01",
                "2023-12-31"
            )
        )
    )


    print(
        "\nSPEI-SR input:"
    )

    print(
        spei
    )


    # Annual mean
    

    print(
        "\nCalculating annual mean SPEI-SR-3..."
    )


    annual = (
        spei
        .resample(
            time="YS"
        )
        .mean(
            dim="time",
            skipna=True
        )
    )


    years = (
        annual.time.dt.year.values
    )


    annual = (
        annual
        .assign_coords(
            year=(
                "time",
                years
            )
        )
        .swap_dims(
            {
                "time":
                    "year"
            }
        )
        .drop_vars(
            "time"
        )
    )


    annual_np = (
        annual
        .values
        .astype(
            np.float64
        )
    )


    year_values = (
        annual.year.values.astype(
            int
        )
    )


    mask1 = (
        (year_values >= P1_START)
        &
        (year_values <= P1_END)
    )


    mask2 = (
        (year_values >= P2_START)
        &
        (year_values <= P2_END)
    )


    print(
        "\nPeriod 1:",
        year_values[
            mask1
        ][0],
        "-",
        year_values[
            mask1
        ][-1],
        "| N =",
        mask1.sum()
    )


    print(
        "Period 2:",
        year_values[
            mask2
        ][0],
        "-",
        year_values[
            mask2
        ][-1],
        "| N =",
        mask2.sum()
    )


    
    # Trends
   

    print(
        "\nPeriod 1 SPEI-SR trend..."
    )

    r1 = prais_winsten_trend(
        annual_np[
            mask1
        ],
        year_values[
            mask1
        ],
        MIN_YEARS_P1
    )


    print(
        "\nPeriod 2 SPEI-SR trend..."
    )

    r2 = prais_winsten_trend(
        annual_np[
            mask2
        ],
        year_values[
            mask2
        ],
        MIN_YEARS_P2
    )


    
    # FDR
    

    print(
        "\nApplying FDR..."
    )


    q1 = bh_fdr_qvalues(
        r1[
            "pvalue"
        ]
    )


    q2 = bh_fdr_qvalues(
        r2[
            "pvalue"
        ]
    )


    
    # SPEI trend dataset
   

    srtrend = xr.Dataset(
        {
            "slope_1979_2000":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    r1[
                        "slope"
                    ].astype(
                        np.float32
                    )
                ),

            "pvalue_1979_2000":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    r1[
                        "pvalue"
                    ].astype(
                        np.float32
                    )
                ),

            "qvalue_1979_2000":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    q1.astype(
                        np.float32
                    )
                ),

            "rho_1979_2000":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    r1[
                        "rho"
                    ].astype(
                        np.float32
                    )
                ),

            "slope_2001_2023":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    r2[
                        "slope"
                    ].astype(
                        np.float32
                    )
                ),

            "pvalue_2001_2023":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    r2[
                        "pvalue"
                    ].astype(
                        np.float32
                    )
                ),

            "qvalue_2001_2023":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    q2.astype(
                        np.float32
                    )
                ),

            "rho_2001_2023":
                (
                    (
                        "lat",
                        "lon"
                    ),
                    r2[
                        "rho"
                    ].astype(
                        np.float32
                    )
                ),
        },

        coords={
            "lat":
                annual.lat.values,

            "lon":
                annual.lon.values,
        }
    )


    srtrend.attrs.update(
        {
            "title":
                "Corrected SPEI-SR-3 annual trend diagnostics",

            "interpretation":
                "Standardized storage-change residual tendency",

            "trend_method":
                "Prais-Winsten AR1 regression",

            "period_1":
                "1979-2000",

            "period_2":
                "2001-2023",
        }
    )


    encoding = {}

    for var in srtrend.data_vars:

        encoding[
            var
        ] = {
            "dtype":
                "float32",

            "zlib":
                True,

            "complevel":
                4,

            "_FillValue":
                np.float32(
                    -9999.0
                ),
        }


    srtrend.to_netcdf(
        SPEI_TREND_CACHE,
        encoding=encoding
    )


    print(
        "\nSaved SPEI-SR trend cache:"
    )

    print(
        SPEI_TREND_CACHE
    )


    spei_ds.close()

    del annual_np
    del r1
    del r2

    gc.collect()


else:

    print(
        "\nUsing existing SPEI-SR trend cache:"
    )

    print(
        SPEI_TREND_CACHE
    )




print("\n" + "=" * 80)
print("LOADING SPEI-SR AND DIRECT STATE TREND PRODUCTS")
print("=" * 80)


sr_ds = xr.open_dataset(
    SPEI_TREND_CACHE
)

state_ds = xr.open_dataset(
    STATE_TREND_FP
)

area_ds = xr.open_dataset(
    LAND_AREA_FP
)



# SPEI-SR


sr_s1 = sr_ds[
    "slope_1979_2000"
]

sr_s2 = sr_ds[
    "slope_2001_2023"
]

sr_p1 = sr_ds[
    "pvalue_1979_2000"
]

sr_p2 = sr_ds[
    "pvalue_2001_2023"
]

sr_q1 = sr_ds[
    "qvalue_1979_2000"
]

sr_q2 = sr_ds[
    "qvalue_2001_2023"
]



# Direct soil+SWE state


st_s1 = state_ds[
    "slope_1979_2000"
]

st_s2 = state_ds[
    "slope_2001_2023"
]

st_p1 = state_ds[
    "pvalue_1979_2000"
]

st_p2 = state_ds[
    "pvalue_2001_2023"
]

st_q1 = state_ds[
    "qvalue_fdr_1979_2000"
]

st_q2 = state_ds[
    "qvalue_fdr_2001_2023"
]


land_area = area_ds[
    "actual_land_area"
]



# 8. GRID CHECK AND ALIGNMENT


if (
    sr_s1.sizes[
        "lat"
    ]
    !=
    st_s1.sizes[
        "lat"
    ]
    or
    sr_s1.sizes[
        "lon"
    ]
    !=
    st_s1.sizes[
        "lon"
    ]
):

    raise RuntimeError(
        "SPEI-SR and direct-state grid dimensions differ."
    )


if not np.allclose(
    sr_s1.lat.values,
    st_s1.lat.values,
    atol=1e-7
):

    raise RuntimeError(
        "Latitude coordinates differ."
    )


if not np.allclose(
    sr_s1.lon.values,
    st_s1.lon.values,
    atol=1e-7
):

    raise RuntimeError(
        "Longitude coordinates differ."
    )


# Force identical coordinates
for da_tmp in [
    sr_s1,
    sr_s2,
    sr_p1,
    sr_p2,
    sr_q1,
    sr_q2,
]:

    da_tmp.coords[
        "lat"
    ] = st_s1.lat

    da_tmp.coords[
        "lon"
    ] = st_s1.lon


(
    sr_s1,
    sr_s2,
    sr_p1,
    sr_p2,
    sr_q1,
    sr_q2,
    st_s1,
    st_s2,
    st_p1,
    st_p2,
    st_q1,
    st_q2,
    land_area,
) = xr.align(
    sr_s1,
    sr_s2,
    sr_p1,
    sr_p2,
    sr_q1,
    sr_q2,
    st_s1,
    st_s2,
    st_p1,
    st_p2,
    st_q1,
    st_q2,
    land_area,
    join="exact"
)


lat = st_s1.lat.values
lon = st_s1.lon.values


# COMMON VALID LAND


valid = (
    np.isfinite(
        sr_s1
    )
    &
    np.isfinite(
        sr_s2
    )
    &
    np.isfinite(
        st_s1
    )
    &
    np.isfinite(
        st_s2
    )
    &
    np.isfinite(
        land_area
    )
    &
    (
        land_area > 0
    )
)


TOTAL_AREA = float(
    land_area.where(
        valid
    ).sum(
        skipna=True
    )
)


print(
    "\nCommon valid land area:"
)

print(
    f"{TOTAL_AREA / 1e12:.3f} million km²"
)



# WORLD BOUNDARY


world = None

if WORLD_SHP.exists():

    print(
        "\nLoading world shapefile..."
    )

    world = (
        gpd.read_file(
            WORLD_SHP
        )
        .to_crs(
            "EPSG:4326"
        )
    )


    world = world[
        world.geometry.notna()
        &
        (~world.geometry.is_empty)
    ].copy()

else:

    print(
        "\nWorld shapefile not found. "
        "Cartopy coastlines will be used."
    )



# AREA FUNCTIONS


def area_of(
    mask
):

    return float(
        land_area.where(
            mask
        ).sum(
            skipna=True
        )
    )


def pct_total(
    mask
):

    return (
        100.0
        *
        area_of(
            mask
        )
        /
        TOTAL_AREA
    )


# AREA-WEIGHTED CORRELATION


def weighted_corr(
    x,
    y,
    w
):

    x = np.asarray(
        x,
        dtype=np.float64
    ).ravel()

    y = np.asarray(
        y,
        dtype=np.float64
    ).ravel()

    w = np.asarray(
        w,
        dtype=np.float64
    ).ravel()


    good = (
        np.isfinite(
            x
        )
        &
        np.isfinite(
            y
        )
        &
        np.isfinite(
            w
        )
        &
        (
            w > 0
        )
    )


    x = x[
        good
    ]

    y = y[
        good
    ]

    w = w[
        good
    ]


    sw = np.sum(
        w
    )


    mx = np.sum(
        w
        *
        x
    ) / sw


    my = np.sum(
        w
        *
        y
    ) / sw


    cov = np.sum(
        w
        *
        (
            x - mx
        )
        *
        (
            y - my
        )
    ) / sw


    vx = np.sum(
        w
        *
        (
            x - mx
        ) ** 2
    ) / sw


    vy = np.sum(
        w
        *
        (
            y - my
        ) ** 2
    ) / sw


    if (
        vx <= 0
        or
        vy <= 0
    ):

        return np.nan


    return (
        cov
        /
        np.sqrt(
            vx
            *
            vy
        )
    )




def build_period1_class(
    slope,
    sig
):
    """
    0 = uncertain
    1 = positive
    2 = negative
    """

    output = xr.full_like(
        slope,
        np.nan,
        dtype=np.float32
    )


    output = xr.where(
        valid,
        0.0,
        output
    )


    output = xr.where(
        valid
        &
        sig
        &
        (slope > 0),
        1.0,
        output
    )


    output = xr.where(
        valid
        &
        sig
        &
        (slope < 0),
        2.0,
        output
    )


    return output


#  CLASSIFICATION


def build_sequence_class(
    slope1,
    slope2,
    sig1,
    sig2
):
    """
    0 = uncertain
    1 = negative -> negative
    2 = positive -> positive
    3 = positive -> negative
    4 = negative -> positive
    """

    robust = (
        valid
        &
        sig1
        &
        sig2
    )


    output = xr.full_like(
        slope1,
        np.nan,
        dtype=np.float32
    )


    output = xr.where(
        valid,
        0.0,
        output
    )


    # negative -> negative
    output = xr.where(
        robust
        &
        (slope1 < 0)
        &
        (slope2 < 0),
        1.0,
        output
    )


    # positive -> positive
    output = xr.where(
        robust
        &
        (slope1 > 0)
        &
        (slope2 > 0),
        2.0,
        output
    )


    # positive -> negative
    output = xr.where(
        robust
        &
        (slope1 > 0)
        &
        (slope2 < 0),
        3.0,
        output
    )


    # negative -> positive
    output = xr.where(
        robust
        &
        (slope1 < 0)
        &
        (slope2 > 0),
        4.0,
        output
    )


    return (
        output,
        robust
    )



# SIGN-ONLY CLASSIFICATION


def sign_sequence(
    slope1,
    slope2
):
    """
    1 = negative -> negative
    2 = positive -> positive
    3 = positive -> negative
    4 = negative -> positive
    """

    output = xr.full_like(
        slope1,
        np.nan,
        dtype=np.float32
    )


    good = (
        valid
        &
        np.isfinite(
            slope1
        )
        &
        np.isfinite(
            slope2
        )
        &
        (slope1 != 0)
        &
        (slope2 != 0)
    )


    output = xr.where(
        good
        &
        (slope1 < 0)
        &
        (slope2 < 0),
        1.0,
        output
    )


    output = xr.where(
        good
        &
        (slope1 > 0)
        &
        (slope2 > 0),
        2.0,
        output
    )


    output = xr.where(
        good
        &
        (slope1 > 0)
        &
        (slope2 < 0),
        3.0,
        output
    )


    output = xr.where(
        good
        &
        (slope1 < 0)
        &
        (slope2 > 0),
        4.0,
        output
    )


    return output



BASE_CMAP = ListedColormap(
    [
        "#D9D9D9",
        "#4393C3",
        "#D6604D",
    ]
)


BASE_NORM = mcolors.BoundaryNorm(
    [
        -0.5,
        0.5,
        1.5,
        2.5
    ],
    BASE_CMAP.N
)


SEQ_CMAP = ListedColormap(
    [
        "#D9D9D9",
        "#D6604D",
        "#4393C3",
        "#FDAE61",
        "#66BD63",
    ]
)


SEQ_NORM = mcolors.BoundaryNorm(
    [
        -0.5,
        0.5,
        1.5,
        2.5,
        3.5,
        4.5
    ],
    SEQ_CMAP.N
)


AGREE_CMAP = ListedColormap(
    [
        "#E5E5E5",
        "#CAB2D6",
        "#B2DF8A",
        "#1B9E77",
        "#D95F02",
    ]
)


AGREE_NORM = mcolors.BoundaryNorm(
    [
        -0.5,
        0.5,
        1.5,
        2.5,
        3.5,
        4.5
    ],
    AGREE_CMAP.N
)



def style_map(
    ax
):

    ax.set_global()

    ax.set_facecolor(
        "white"
    )


    ax.add_feature(
        cfeature.OCEAN,
        facecolor="white",
        zorder=0
    )


    if world is not None:

        ax.add_geometries(
            world.geometry,
            crs=ccrs.PlateCarree(),
            facecolor="none",
            edgecolor="0.25",
            linewidth=0.30,
            zorder=5
        )

    else:

        ax.coastlines(
            resolution="110m",
            linewidth=0.45,
            zorder=5
        )


    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.30,
        linestyle="--",
        color="0.5",
        alpha=0.40
    )


    gl.top_labels = False
    gl.right_labels = False


    gl.xlabel_style = {
        "size":
            8
    }

    gl.ylabel_style = {
        "size":
            8
    }



def analyze_and_plot(
    criterion_name,
    sr_sig1,
    sr_sig2,
    st_sig1,
    st_sig2,
    suffix
):

    print("\n" + "=" * 80)
    print(criterion_name)
    print("=" * 80)




    sr_sig1 = (
        sr_sig1
        &
        valid
    )

    sr_sig2 = (
        sr_sig2
        &
        valid
    )


    st_sig1 = (
        st_sig1
        &
        valid
    )

    st_sig2 = (
        st_sig2
        &
        valid
    )




    sr_base = build_period1_class(
        sr_s1,
        sr_sig1
    )


    st_base = build_period1_class(
        st_s1,
        st_sig1
    )




    (
        sr_class,
        sr_robust
    ) = build_sequence_class(
        sr_s1,
        sr_s2,
        sr_sig1,
        sr_sig2
    )


    (
        st_class,
        st_robust
    ) = build_sequence_class(
        st_s1,
        st_s2,
        st_sig1,
        st_sig2
    )




    sr_sign_class = sign_sequence(
        sr_s1,
        sr_s2
    )


    st_sign_class = sign_sequence(
        st_s1,
        st_s2
    )




    sr_p1_pct = {
        "uncertain":
            pct_total(
                valid
                &
                (~sr_sig1)
            ),

        "positive":
            pct_total(
                sr_sig1
                &
                (sr_s1 > 0)
            ),

        "negative":
            pct_total(
                sr_sig1
                &
                (sr_s1 < 0)
            ),
    }


    st_p1_pct = {
        "uncertain":
            pct_total(
                valid
                &
                (~st_sig1)
            ),

        "positive":
            pct_total(
                st_sig1
                &
                (st_s1 > 0)
            ),

        "negative":
            pct_total(
                st_sig1
                &
                (st_s1 < 0)
            ),
    }




    def sequence_percentages(
        cls
    ):

        return {
            "uncertain":
                pct_total(
                    valid
                    &
                    (cls == 0)
                ),

            "NN":
                pct_total(
                    cls == 1
                ),

            "PP":
                pct_total(
                    cls == 2
                ),

            "PN":
                pct_total(
                    cls == 3
                ),

            "NP":
                pct_total(
                    cls == 4
                ),
        }


    sr_pct = sequence_percentages(
        sr_class
    )


    st_pct = sequence_percentages(
        st_class
    )




    p1_same_sign = (
        valid
        &
        (
            (
                (sr_s1 > 0)
                &
                (st_s1 > 0)
            )
            |
            (
                (sr_s1 < 0)
                &
                (st_s1 < 0)
            )
        )
    )


    p2_same_sign = (
        valid
        &
        (
            (
                (sr_s2 > 0)
                &
                (st_s2 > 0)
            )
            |
            (
                (sr_s2 < 0)
                &
                (st_s2 < 0)
            )
        )
    )




    common_sign_valid = (
        valid
        &
        np.isfinite(
            sr_sign_class
        )
        &
        np.isfinite(
            st_sign_class
        )
    )


    common_sign_area = area_of(
        common_sign_valid
    )


    if common_sign_area <= 0:

        raise RuntimeError(
            "No common valid area available for sign-pattern comparison."
        )


    all_sequence_agree = (
        common_sign_valid
        &
        (
            sr_sign_class
            ==
            st_sign_class
        )
    )


    sign_pattern_agreement = (
        100.0
        *
        area_of(
            all_sequence_agree
        )
        /
        common_sign_area
    )


    print(
        "\nArea available for sign-pattern confusion matrix:"
    )

    print(
        f"{common_sign_area / 1e12:.3f} million km²"
    )




    common_robust = (
        sr_robust
        &
        st_robust
    )


    union_robust = (
        sr_robust
        |
        st_robust
    )


    common_agree = (
        common_robust
        &
        (
            sr_class
            ==
            st_class
        )
    )


    common_disagree = (
        common_robust
        &
        (
            sr_class
            !=
            st_class
        )
    )


    sr_robust_area = area_of(
        sr_robust
    )


    st_robust_area = area_of(
        st_robust
    )


    common_robust_area = area_of(
        common_robust
    )


    union_robust_area = area_of(
        union_robust
    )


    if union_robust_area > 0:

        robust_jaccard = (
            100.0
            *
            common_robust_area
            /
            union_robust_area
        )

    else:

        robust_jaccard = np.nan


    if common_robust_area > 0:

        robust_class_agreement = (
            100.0
            *
            area_of(
                common_agree
            )
            /
            common_robust_area
        )

    else:

        robust_class_agreement = np.nan




    corr_p1 = weighted_corr(
        sr_s1.values,
        st_s1.values,
        land_area.values
    )


    corr_p2 = weighted_corr(
        sr_s2.values,
        st_s2.values,
        land_area.values
    )



    # AREA-WEIGHTED SIGN-PATTERN CONFUSION MATRIX


    confusion_area = np.zeros(
        (
            4,
            4
        ),
        dtype=np.float64
    )


    for i in range(
        1,
        5
    ):

        for j in range(
            1,
            5
        ):

            mask = (
                common_sign_valid
                &
                (
                    sr_sign_class
                    ==
                    i
                )
                &
                (
                    st_sign_class
                    ==
                    j
                )
            )


            confusion_area[
                i - 1,
                j - 1
            ] = area_of(
                mask
            )


    confusion_pct = (
        100.0
        *
        confusion_area
        /
        common_sign_area
    )


    print(
        "\nArea-weighted sign-pattern confusion matrix (%):"
    )

    print(
        np.round(
            confusion_pct,
            3
        )
    )


    
    # AREA-WEIGHTED COHEN KAPPA
   

    probability_matrix = (
        confusion_area
        /
        common_sign_area
    )


    observed_agreement = np.trace(
        probability_matrix
    )


    row_probability = np.sum(
        probability_matrix,
        axis=1
    )


    column_probability = np.sum(
        probability_matrix,
        axis=0
    )


    expected_agreement = np.sum(
        row_probability
        *
        column_probability
    )


    if (
        np.isfinite(
            expected_agreement
        )
        and
        expected_agreement < 1.0
    ):

        kappa_sign = (
            observed_agreement
            -
            expected_agreement
        ) / (
            1.0
            -
            expected_agreement
        )

    else:

        kappa_sign = np.nan


    print(
        "\nSIGN-PATTERN AGREEMENT"
    )

    print(
        f"Observed agreement = "
        f"{100.0 * observed_agreement:.3f}%"
    )

    print(
        f"Area-weighted Cohen kappa = "
        f"{kappa_sign:.4f}"
    )




    agreement_map = xr.full_like(
        sr_s1,
        np.nan,
        dtype=np.float32
    )


    # All valid land begins as "neither robust"
    agreement_map = xr.where(
        valid,
        0.0,
        agreement_map
    )


    # SPEI-SR only robust
    agreement_map = xr.where(
        sr_robust
        &
        (~st_robust),
        1.0,
        agreement_map
    )


    # direct state only robust
    agreement_map = xr.where(
        (~sr_robust)
        &
        st_robust,
        2.0,
        agreement_map
    )


    # both robust and same class
    agreement_map = xr.where(
        common_agree,
        3.0,
        agreement_map
    )


    # both robust but different class
    agreement_map = xr.where(
        common_disagree,
        4.0,
        agreement_map
    )


    agreement_pct = {
        "neither":
            pct_total(
                agreement_map
                ==
                0
            ),

        "sr_only":
            pct_total(
                agreement_map
                ==
                1
            ),

        "state_only":
            pct_total(
                agreement_map
                ==
                2
            ),

        "both_agree":
            pct_total(
                agreement_map
                ==
                3
            ),

        "both_disagree":
            pct_total(
                agreement_map
                ==
                4
            ),
    }



    # ========================================================

    summary = {
        "criterion":
            criterion_name,

        "Period1_sign_agreement_pct_total_land":
            pct_total(
                p1_same_sign
            ),

        "Period2_sign_agreement_pct_total_land":
            pct_total(
                p2_same_sign
            ),

        "Four_pattern_sign_agreement_pct_common_valid":
            sign_pattern_agreement,

        "SPEI_SR_robust_pct_total_land":
            (
                100.0
                *
                sr_robust_area
                /
                TOTAL_AREA
            ),

        "Direct_state_robust_pct_total_land":
            (
                100.0
                *
                st_robust_area
                /
                TOTAL_AREA
            ),

        "Common_robust_pct_total_land":
            (
                100.0
                *
                common_robust_area
                /
                TOTAL_AREA
            ),

        "Robust_mask_Jaccard_pct":
            robust_jaccard,

        "Common_robust_class_agreement_pct":
            robust_class_agreement,

        "Sign_pattern_agreement_pct":
            (
                100.0
                *
                observed_agreement
            ),

        "Area_weighted_Cohen_kappa_sign_patterns":
            kappa_sign,

        "Weighted_slope_correlation_P1":
            corr_p1,

        "Weighted_slope_correlation_P2":
            corr_p2,
    }


    print(
        "\nAGREEMENT DIAGNOSTICS"
    )

    print(
        "-" * 80
    )


    for key, value in summary.items():

        if key == "criterion":
            continue

        print(
            f"{key}: "
            f"{value:.4f}"
        )



    SUMMARY_CSV = (
        OUT_DIR /
        f"comparison_agreement_summary_{suffix}.csv"
    )


    pd.DataFrame(
        [
            summary
        ]
    ).to_csv(
        SUMMARY_CSV,
        index=False
    )



    rows = []


    for key, value in sr_p1_pct.items():

        rows.append(
            {
                "diagnostic":
                    "Corrected SPEI-SR Period 1",

                "class":
                    key,

                "percent_total_valid_land":
                    value,
            }
        )


    for key, value in st_p1_pct.items():

        rows.append(
            {
                "diagnostic":
                    "Direct soil+SWE Period 1",

                "class":
                    key,

                "percent_total_valid_land":
                    value,
            }
        )


    for key, value in sr_pct.items():

        rows.append(
            {
                "diagnostic":
                    "Corrected SPEI-SR sequence",

                "class":
                    key,

                "percent_total_valid_land":
                    value,
            }
        )


    for key, value in st_pct.items():

        rows.append(
            {
                "diagnostic":
                    "Direct soil+SWE transition",

                "class":
                    key,

                "percent_total_valid_land":
                    value,
            }
        )


    for key, value in agreement_pct.items():

        rows.append(
            {
                "diagnostic":
                    "Robustness agreement",

                "class":
                    key,

                "percent_total_valid_land":
                    value,
            }
        )


    CLASS_CSV = (
        OUT_DIR /
        f"comparison_area_percentages_{suffix}.csv"
    )


    pd.DataFrame(
        rows
    ).to_csv(
        CLASS_CSV,
        index=False
    )



    residual_labels = [
        "Rminus_to_Rminus",
        "Rplus_to_Rplus",
        "Rplus_to_Rminus",
        "Rminus_to_Rplus",
    ]


    state_labels = [
        "DD",
        "WW",
        "WD",
        "DW",
    ]


    confusion_rows = []


    for i in range(
        4
    ):

        for j in range(
            4
        ):

            confusion_rows.append(
                {
                    "SPEI_SR_pattern":
                        residual_labels[
                            i
                        ],

                    "Direct_state_pattern":
                        state_labels[
                            j
                        ],

                    "area_km2":
                        confusion_area[
                            i,
                            j
                        ]
                        /
                        1e6,

                    "percent_common_valid_land":
                        confusion_pct[
                            i,
                            j
                        ],

                    "percent_total_valid_land":
                        (
                            100.0
                            *
                            confusion_area[
                                i,
                                j
                            ]
                            /
                            TOTAL_AREA
                        ),
                }
            )


    CONFUSION_CSV = (
        OUT_DIR /
        f"area_weighted_sign_pattern_confusion_{suffix}.csv"
    )


    pd.DataFrame(
        confusion_rows
    ).to_csv(
        CONFUSION_CSV,
        index=False
    )


  
    # SAVE COMPARISON NETCDF
   

    COMPARE_NC = (
        OUT_DIR /
        f"SPEI_SR_vs_direct_state_comparison_{suffix}.nc"
    )


    comparison_ds = xr.Dataset(
        {
            "SPEI_SR_robust_class":
                sr_class.astype(
                    np.float32
                ),

            "direct_state_robust_class":
                st_class.astype(
                    np.float32
                ),

            "SPEI_SR_sign_pattern":
                sr_sign_class.astype(
                    np.float32
                ),

            "direct_state_sign_pattern":
                st_sign_class.astype(
                    np.float32
                ),

            "robustness_agreement_category":
                agreement_map.astype(
                    np.float32
                ),

            "SPEI_SR_slope_1979_2000":
                sr_s1.astype(
                    np.float32
                ),

            "SPEI_SR_slope_2001_2023":
                sr_s2.astype(
                    np.float32
                ),

            "state_slope_1979_2000":
                st_s1.astype(
                    np.float32
                ),

            "state_slope_2001_2023":
                st_s2.astype(
                    np.float32
                ),
        }
    )


    comparison_ds.attrs.update(
        {
            "criterion":
                criterion_name,

            "important_note":
                (
                    "SPEI-SR patterns describe trends in a "
                    "standardized storage-change residual. "
                    "Direct soil+SWE patterns describe trends "
                    "in modeled storage state."
                )
        }
    )


    encoding = {}

    for var in comparison_ds.data_vars:

        encoding[
            var
        ] = {
            "dtype":
                "float32",

            "zlib":
                True,

            "complevel":
                4,

            "_FillValue":
                np.float32(
                    -9999.0
                ),
        }


    comparison_ds.to_netcdf(
        COMPARE_NC,
        encoding=encoding
    )


  
    #FIGURE
  

    print(
        "\nCreating six-panel comparison figure..."
    )


    fig = plt.figure(
        figsize=(
            17,
            17
        )
    )


    proj = ccrs.PlateCarree()


    ax1 = fig.add_subplot(
        3,
        2,
        1,
        projection=proj
    )

    ax2 = fig.add_subplot(
        3,
        2,
        2,
        projection=proj
    )

    ax3 = fig.add_subplot(
        3,
        2,
        3,
        projection=proj
    )

    ax4 = fig.add_subplot(
        3,
        2,
        4,
        projection=proj
    )

    ax5 = fig.add_subplot(
        3,
        2,
        5,
        projection=proj
    )

    ax6 = fig.add_subplot(
        3,
        2,
        6
    )


    for ax in [
        ax1,
        ax2,
        ax3,
        ax4,
        ax5,
    ]:

        style_map(
            ax
        )




    im1 = ax1.pcolormesh(
        lon,
        lat,
        sr_base.values,
        cmap=BASE_CMAP,
        norm=BASE_NORM,
        shading="auto",
        transform=proj,
        rasterized=True,
        zorder=1
    )


    ax1.set_title(
        "(a) Corrected SPEI-SR-3 residual tendency, 1979–2000",
        fontsize=12,
        fontweight="bold"
    )


    cb1 = fig.colorbar(
        im1,
        ax=ax1,
        orientation="horizontal",
        fraction=0.050,
        pad=0.055,
        ticks=[
            0,
            1,
            2
        ]
    )


    cb1.set_ticklabels(
        [
            (
                "Uncertain\n"
                f"{sr_p1_pct['uncertain']:.1f}%"
            ),
            (
                "Positive residual trend\n"
                f"{sr_p1_pct['positive']:.1f}%"
            ),
            (
                "Negative residual trend\n"
                f"{sr_p1_pct['negative']:.1f}%"
            ),
        ]
    )


    cb1.ax.tick_params(
        labelsize=8
    )




    im2 = ax2.pcolormesh(
        lon,
        lat,
        st_base.values,
        cmap=BASE_CMAP,
        norm=BASE_NORM,
        shading="auto",
        transform=proj,
        rasterized=True,
        zorder=1
    )


    ax2.set_title(
        "(b) Direct soil + SWE storage-state trend, 1979–2000",
        fontsize=12,
        fontweight="bold"
    )


    cb2 = fig.colorbar(
        im2,
        ax=ax2,
        orientation="horizontal",
        fraction=0.050,
        pad=0.055,
        ticks=[
            0,
            1,
            2
        ]
    )


    cb2.set_ticklabels(
        [
            (
                "Uncertain\n"
                f"{st_p1_pct['uncertain']:.1f}%"
            ),
            (
                "Wetting\n"
                f"{st_p1_pct['positive']:.1f}%"
            ),
            (
                "Drying\n"
                f"{st_p1_pct['negative']:.1f}%"
            ),
        ]
    )


    cb2.ax.tick_params(
        labelsize=8
    )




    im3 = ax3.pcolormesh(
        lon,
        lat,
        sr_class.values,
        cmap=SEQ_CMAP,
        norm=SEQ_NORM,
        shading="auto",
        transform=proj,
        rasterized=True,
        zorder=1
    )


    ax3.set_title(
        "(c) Corrected SPEI-SR residual-tendency sequences",
        fontsize=12,
        fontweight="bold"
    )


    cb3 = fig.colorbar(
        im3,
        ax=ax3,
        orientation="horizontal",
        fraction=0.050,
        pad=0.055,
        ticks=[
            0,
            1,
            2,
            3,
            4
        ]
    )


    cb3.set_ticklabels(
        [
            (
                "Uncertain\n"
                f"{sr_pct['uncertain']:.1f}%"
            ),
            (
                "R−→R−\n"
                f"{sr_pct['NN']:.1f}%"
            ),
            (
                "R+→R+\n"
                f"{sr_pct['PP']:.1f}%"
            ),
            (
                "R+→R−\n"
                f"{sr_pct['PN']:.1f}%"
            ),
            (
                "R−→R+\n"
                f"{sr_pct['NP']:.1f}%"
            ),
        ]
    )


    cb3.ax.tick_params(
        labelsize=8
    )




    im4 = ax4.pcolormesh(
        lon,
        lat,
        st_class.values,
        cmap=SEQ_CMAP,
        norm=SEQ_NORM,
        shading="auto",
        transform=proj,
        rasterized=True,
        zorder=1
    )


    ax4.set_title(
        "(d) Direct soil + SWE storage-state transitions",
        fontsize=12,
        fontweight="bold"
    )


    cb4 = fig.colorbar(
        im4,
        ax=ax4,
        orientation="horizontal",
        fraction=0.050,
        pad=0.055,
        ticks=[
            0,
            1,
            2,
            3,
            4
        ]
    )


    cb4.set_ticklabels(
        [
            (
                "Uncertain\n"
                f"{st_pct['uncertain']:.1f}%"
            ),
            (
                "DD\n"
                f"{st_pct['NN']:.1f}%"
            ),
            (
                "WW\n"
                f"{st_pct['PP']:.1f}%"
            ),
            (
                "WD\n"
                f"{st_pct['PN']:.1f}%"
            ),
            (
                "DW\n"
                f"{st_pct['NP']:.1f}%"
            ),
        ]
    )


    cb4.ax.tick_params(
        labelsize=8
    )




    im5 = ax5.pcolormesh(
        lon,
        lat,
        agreement_map.values,
        cmap=AGREE_CMAP,
        norm=AGREE_NORM,
        shading="auto",
        transform=proj,
        rasterized=True,
        zorder=1
    )


    ax5.set_title(
        "(e) Robustness and class agreement",
        fontsize=12,
        fontweight="bold"
    )


    cb5 = fig.colorbar(
        im5,
        ax=ax5,
        orientation="horizontal",
        fraction=0.050,
        pad=0.055,
        ticks=[
            0,
            1,
            2,
            3,
            4
        ]
    )


    cb5.set_ticklabels(
        [
            (
                "Neither robust\n"
                f"{agreement_pct['neither']:.1f}%"
            ),
            (
                "SPEI-SR only\n"
                f"{agreement_pct['sr_only']:.1f}%"
            ),
            (
                "State only\n"
                f"{agreement_pct['state_only']:.1f}%"
            ),
            (
                "Both agree\n"
                f"{agreement_pct['both_agree']:.1f}%"
            ),
            (
                "Both disagree\n"
                f"{agreement_pct['both_disagree']:.1f}%"
            ),
        ]
    )


    cb5.ax.tick_params(
        labelsize=8
    )



    plot_matrix = np.ma.masked_invalid(
        confusion_pct
    )


    matrix_max = float(
        np.nanmax(
            confusion_pct
        )
    )


    if (
        not np.isfinite(
            matrix_max
        )
        or
        matrix_max <= 0
    ):

        matrix_max = 1.0


    im6 = ax6.imshow(
        plot_matrix,
        vmin=0,
        vmax=matrix_max,
        aspect="equal",
        interpolation="nearest"
    )


    ax6.set_xticks(
        np.arange(
            4
        )
    )


    ax6.set_yticks(
        np.arange(
            4
        )
    )


    ax6.set_xticklabels(
        [
            "DD",
            "WW",
            "WD",
            "DW"
        ],
        fontsize=10
    )


    ax6.set_yticklabels(
        [
            "R−→R−",
            "R+→R+",
            "R+→R−",
            "R−→R+"
        ],
        fontsize=10
    )


    ax6.set_xlabel(
        "Direct soil + SWE state-trend pattern",
        fontsize=11
    )


    ax6.set_ylabel(
        "Corrected SPEI-SR residual-tendency pattern",
        fontsize=11
    )


    ax6.set_title(
        "(f) Area-weighted sign-pattern comparison\n"
        f"Agreement = {100.0 * observed_agreement:.1f}%, "
        f"κ = {kappa_sign:.2f}",
        fontsize=12,
        fontweight="bold"
    )




    text_threshold = (
        matrix_max
        *
        0.50
    )


    for i in range(
        4
    ):

        for j in range(
            4
        ):

            value = confusion_pct[
                i,
                j
            ]


            if np.isfinite(
                value
            ):

                text_color = (
                    "white"
                    if value > text_threshold
                    else
                    "black"
                )


                ax6.text(
                    j,
                    i,
                    f"{value:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=10,
                    fontweight="bold",
                    color=text_color
                )


    cbar6 = fig.colorbar(
        im6,
        ax=ax6,
        fraction=0.046,
        pad=0.04
    )


    cbar6.set_label(
        "Percentage of common valid land",
        fontsize=10
    )




    fig.suptitle(
        "Corrected SPEI-SR storage-change tendencies versus "
        "direct soil + SWE storage-state trends\n"
        + criterion_name,
        fontsize=16,
        fontweight="bold",
        y=0.995
    )


    # ========================================================
    # BOTTOM DIAGNOSTIC TEXT
    # ========================================================

    diagnostic_text = (
        f"P1 sign agreement = "
        f"{summary['Period1_sign_agreement_pct_total_land']:.1f}%   |   "
        f"P2 sign agreement = "
        f"{summary['Period2_sign_agreement_pct_total_land']:.1f}%   |   "
        f"4-pattern agreement = "
        f"{summary['Sign_pattern_agreement_pct']:.1f}%   |   "
        f"κ = {kappa_sign:.2f}   |   "
        f"Common robust area = "
        f"{summary['Common_robust_pct_total_land']:.1f}%   |   "
        f"Robust Jaccard = "
        f"{robust_jaccard:.1f}%   |   "
        f"Weighted slope r: "
        f"P1={corr_p1:.2f}, P2={corr_p2:.2f}"
    )


    fig.text(
        0.5,
        0.012,
        diagnostic_text,
        ha="center",
        va="bottom",
        fontsize=9.2
    )


    plt.tight_layout(
        rect=[
            0.015,
            0.035,
            0.985,
            0.965
        ],
        h_pad=2.1,
        w_pad=1.6
    )


    
    # SAVE FIGURE
   

    OUT_BASE = (
        OUT_DIR /
        f"Combined_SPEI_SR_vs_Direct_State_{suffix}"
    )


    fig.savefig(
        str(
            OUT_BASE
        )
        + ".png",
        dpi=DPI,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        str(
            OUT_BASE
        )
        + ".svg",
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        str(
            OUT_BASE
        )
        + ".pdf",
        dpi=DPI,
        bbox_inches="tight",
        facecolor="white"
    )


    plt.show()

    plt.close(
        fig
    )


  
    # OUTPUT SUMMARY


    print("\n" + "-" * 80)

    print(
        "SAVED FIGURES"
    )

    print(
        str(
            OUT_BASE
        )
        + ".png"
    )

    print(
        str(
            OUT_BASE
        )
        + ".svg"
    )

    print(
        str(
            OUT_BASE
        )
        + ".pdf"
    )


    print(
        "\nSAVED TABLES"
    )

    print(
        SUMMARY_CSV
    )

    print(
        CLASS_CSV
    )

    print(
        CONFUSION_CSV
    )


    print(
        "\nSAVED NETCDF"
    )

    print(
        COMPARE_NC
    )




analyze_and_plot(
    criterion_name=
        "AR(1)-corrected p < 0.05",

    sr_sig1=
        (
            np.isfinite(
                sr_p1
            )
            &
            (
                sr_p1
                <
                ALPHA
            )
        ),

    sr_sig2=
        (
            np.isfinite(
                sr_p2
            )
            &
            (
                sr_p2
                <
                ALPHA
            )
        ),

    st_sig1=
        (
            np.isfinite(
                st_p1
            )
            &
            (
                st_p1
                <
                ALPHA
            )
        ),

    st_sig2=
        (
            np.isfinite(
                st_p2
            )
            &
            (
                st_p2
                <
                ALPHA
            )
        ),

    suffix=
        "AR1_p05"
)



analyze_and_plot(
    criterion_name=
        "AR(1)-corrected FDR q < 0.05",

    sr_sig1=
        (
            np.isfinite(
                sr_q1
            )
            &
            (
                sr_q1
                <
                ALPHA
            )
        ),

    sr_sig2=
        (
            np.isfinite(
                sr_q2
            )
            &
            (
                sr_q2
                <
                ALPHA
            )
        ),

    st_sig1=
        (
            np.isfinite(
                st_q1
            )
            &
            (
                st_q1
                <
                ALPHA
            )
        ),

    st_sig2=
        (
            np.isfinite(
                st_q2
            )
            &
            (
                st_q2
                <
                ALPHA
            )
        ),

    suffix=
        "AR1_FDR_q05"
)



sr_ds.close()
state_ds.close()
area_ds.close()


print("\n" + "=" * 80)
print("COMPARISON ANALYSIS COMPLETED")
print("=" * 80)

print(
    "\nOutput directory:"
)

print(
    OUT_DIR
)